# 개인 추가 실습 · 심전도 혼동행렬과 이미지 복원 평가

01의 마지막 셀까지 실행한 뒤, **01의 커널**에서 아래 한 줄을 한 번 실행한다. 다섯 변수를 현재 IPython 프로필에 저장한다. 임시 코드 셀은 실행 후 지워도 된다.

```python
%store x_test x_test_noisy decoded_imgs test_labels preds
```

같은 컴퓨터·사용자·IPython 프로필에서 02를 실행하면 같은 예측과 복원본으로 평가할 수 있다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# 01에서 저장한 원본, 노이즈 사진, 복원본, 심전도 정답과 예측을 읽습니다.
%store -r x_test x_test_noisy decoded_imgs test_labels preds

In [40]:
import pandas as pd

# 정답과 예측에서 True는 '정상', False는 '이상'으로 표시
label_names = {True: "정상", False: "이상"}
actual = pd.Series(test_labels.cpu().numpy(), name="실제 정답").map(label_names)
predicted = pd.Series(preds.cpu().numpy(), name="모델 예측").map(label_names)

# 실제 정답과 모델 예측을 짝지어 각 경우가 몇 건인지 집계
table = pd.crosstab(actual, predicted)

# 행과 열을 모두 '정상 → 이상' 순서로 표시
table.reindex(index=["정상", "이상"], columns=["정상", "이상"], fill_value=0)

모델 예측,정상,이상
실제 정답,,
정상,509,51
이상,4,436


### 심전도 혼동행렬
행은 실제 정답, 열은 모델 예측이다. 정상 기록을 이상으로 판정한 건수와 이상 기록을 정상으로 판정한 건수를 각각 확인한다.

In [ ]:
# 28×28 픽셀의 제곱 오차를 사진마다 평균 냅니다.
squared_error = (x_test - x_test_noisy) ** 2
mse_per_image = squared_error.mean(dim=(1, 2, 3))

# 픽셀 범위가 0~1이므로 최댓값 1을 사용합니다.
psnr_before = 10 * torch.log10(1.0 / mse_per_image)
print(f"노이즈 제거 전 평균 PSNR: {psnr_before.mean().item():.2f} dB")

### 복원 전 PSNR
깨끗한 원본과 노이즈 사진을 비교해 기준 점수를 구한다. 픽셀 오차가 작을수록 PSNR은 높다.

In [ ]:
# 같은 원본과 복원 사진을 비교합니다.
restored_images = torch.from_numpy(decoded_imgs)
mse_after = ((x_test - restored_images) ** 2).mean(dim=(1, 2, 3))
psnr_after = 10 * torch.log10(1.0 / mse_after)

before = psnr_before.mean().item()
after = psnr_after.mean().item()
print(f"복원 전 {before:.2f} dB, 복원 후 {after:.2f} dB, 변화 {after - before:+.2f} dB")

### 복원 전후 PSNR
사진별 PSNR을 구한 뒤 전체 사진의 평균을 비교한다. 변화량이 양수이면 모델을 거친 사진의 평균 PSNR이 올랐다.

In [ ]:
from skimage.metrics import structural_similarity

# 첫 번째 사진의 흑백 채널을 선택합니다.
original = x_test[0, 0].cpu().numpy()
noisy = x_test_noisy[0, 0].cpu().numpy()
restored = decoded_imgs[0, 0]

ssim_before = structural_similarity(original, noisy, data_range=1.0)
ssim_after = structural_similarity(original, restored, data_range=1.0)
print(f"첫 사진 SSIM: {ssim_before:.4f} → {ssim_after:.4f} ({ssim_after - ssim_before:+.4f})")

### 첫 번째 사진의 SSIM
같은 원본을 기준으로 노이즈 입력과 복원본을 각각 비교한다. SSIM이 1에 가까울수록 국소적인 밝기·대비·구조가 비슷하다.

In [ ]:
# 테스트 세트에서 같은 번호의 사진 세 장을 한 쌍씩 비교합니다.
clean_images = x_test[:, 0].cpu().numpy()
noisy_images = x_test_noisy[:, 0].cpu().numpy()
restored_images = decoded_imgs[:, 0]
ssim_before_scores = []
ssim_after_scores = []
for clean, noisy, restored in zip(clean_images, noisy_images, restored_images):
    ssim_before_scores.append(structural_similarity(clean, noisy, data_range=1.0))
    ssim_after_scores.append(structural_similarity(clean, restored, data_range=1.0))

mean_ssim_before = np.mean(ssim_before_scores)
mean_ssim_after = np.mean(ssim_after_scores)
print(f"전체 평균 SSIM: {mean_ssim_before:.4f} → {mean_ssim_after:.4f}")

### 전체 테스트 사진의 SSIM
앞 셀은 첫 번째 사진의 점수다. 여기서는 테스트 사진 10,000장의 점수를 따로 구한 뒤 평균 낸다.

In [ ]:
# 첫 번째 사진에 같은 계산이 적용됐는지 확인합니다.
print(f"한 장 계산: {ssim_after:.4f}")
print(f"전체 계산 중 첫 사진: {ssim_after_scores[0]:.4f}")
print(f"전체 평균: {mean_ssim_after:.4f}")

### 첫 번째 사진과 전체 평균
앞의 두 값은 같은 사진을 채점했으므로 일치한다. 세 번째 값은 모든 사진을 반영한 평균이다.

In [ ]:
# 지표별로 같은 테스트 사진의 복원 전후 평균을 비교합니다.
results = pd.DataFrame({
    "복원 전 평균": [before, mean_ssim_before],
    "복원 후 평균": [after, mean_ssim_after],
}, index=["PSNR (dB)", "SSIM"])
results["변화량"] = results["복원 후 평균"] - results["복원 전 평균"]
results.round(4)

### 평균 성능 비교
각 행을 왼쪽에서 오른쪽으로 읽는다. PSNR과 SSIM은 척도가 다르므로 변화량은 같은 지표 안에서 해석한다.

In [ ]:
# SSIM 변화량이 가장 큰 사진과 가장 작은 사진을 고릅니다.
ssim_change = np.array(ssim_after_scores) - np.array(ssim_before_scores)
indices = [np.argmax(ssim_change), np.argmin(ssim_change)]
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for row, i in enumerate(indices):
    images = [clean_images[i], noisy_images[i], restored_images[i]]
    titles = [f"Original (#{i})", f"Noisy | SSIM {ssim_before_scores[i]:.3f}",
              f"Restored | SSIM {ssim_after_scores[i]:.3f}"]
    for ax, image, title in zip(axes[row], images, titles):
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()

### 변화량이 가장 큰 사례와 작은 사례
각 줄의 원본, 입력, 복원본을 순서대로 보며 같은 부위의 윤곽과 무늬를 비교한다. 윗줄은 변화량이 가장 큰 사진이다.

In [ ]:
# SSIM이 오르거나 떨어진 사진 수와 그 비율을 구합니다.
counts = pd.Series({
    "상승": np.count_nonzero(ssim_change > 0),
    "동일": np.count_nonzero(ssim_change == 0),
    "하락": np.count_nonzero(ssim_change < 0),
}, name="사진 수")
summary = counts.to_frame()
summary["비율 (%)"] = (counts / len(ssim_change) * 100).round(2)
summary

### 개선된 사진의 비율
평균 변화량은 개선 폭을, 상승 비율은 테스트 사진 중 몇 퍼센트가 개선됐는지를 알려 준다.

In [ ]:
# 같은 사진에서 두 지표가 함께 올랐는지 집계합니다.
psnr_change = (psnr_after - psnr_before).cpu().numpy()
psnr_result = np.where(psnr_change > 0, "상승", "동일 또는 하락")
ssim_result = np.where(ssim_change > 0, "상승", "동일 또는 하락")
order = ["상승", "동일 또는 하락"]
comparison = pd.crosstab(
    pd.Series(psnr_result, name="PSNR 변화"),
    pd.Series(ssim_result, name="SSIM 변화")
)
comparison.reindex(index=order, columns=order, fill_value=0)

### 두 지표가 함께 오른 사진
행은 PSNR, 열은 SSIM이다. 왼쪽 위 칸은 두 지표가 모두 상승한 사진 수이고 네 칸의 합은 전체 테스트 사진 수다.